In [1]:
!wget https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv

--2026-07-01 12:00:56--  https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63103 (62K) [text/plain]
Saving to: ‘split_manifest.csv’

split_manifest.csv  100%[===================>]  61.62K  --.-KB/s    in 0.007s  

2026-07-01 12:00:56 (8.49 MB/s) - ‘split_manifest.csv’ saved [63103/63103]



In [2]:
!wget https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv

--2026-07-03 03:07:23--  https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63103 (62K) [text/plain]
Saving to: ‘split_manifest.csv’

split_manifest.csv  100%[===================>]  61.62K  --.-KB/s    in 0.001s  

2026-07-03 03:07:23 (53.6 MB/s) - ‘split_manifest.csv’ saved [63103/63103]



In [3]:
import pandas as pd
manifest = pd.read_csv('split_manifest.csv')
print(manifest.shape)
manifest.head()

(1203, 4)


,video_path,label,source,split
0,YouTube-real/00000.mp4,1,youtube-real,train
1,Celeb-real/id4_0001.mp4,1,celeb-real,train
2,Celeb-real/id10_0007.mp4,1,celeb-real,train
3,YouTube-real/00079.mp4,1,youtube-real,train
4,Celeb-real/id11_0004.mp4,1,celeb-real,train


In [4]:
print(manifest['label'].unique())
print(manifest['split'].unique())
print(manifest['source'].unique())
print(manifest['label'].value_counts())
print(manifest['split'].value_counts())

[1 0]
['train' 'val' 'test']
['youtube-real' 'celeb-real' 'celeb-synthesis']
label
0    795
1    408
Name: count, dtype: int64
split
train    841
test     182
val      180
Name: count, dtype: int64


In [5]:
print(manifest.groupby(['source', 'label']).size())

source           label
celeb-real       1        158
celeb-synthesis  0        795
youtube-real     1        250
dtype: int64


In [6]:
!mkdir -p ~/.kaggle
!echo KGAT_ea2cc5f42eab7752c7e422757de57a92 > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

In [1]:
!pip install -q kaggle
!kaggle datasets download -d niranjana1310/celeb-df-v2-faces

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata


In [8]:
!unzip -q celeb-df-v2-faces.zip -d faces_data
!ls faces_data

test  train  val


In [9]:
!ls faces_data/train

fake  real


In [10]:
!ls faces_data/train/real
!echo "---"
!ls faces_data/train/fake

crf23  crf28  crf35  original
---
crf23  crf28  crf35  original


In [11]:
!ls faces_data/train/real/original | head -10

00000_frame00000.jpg
00000_frame00010.jpg
00000_frame00020.jpg
00000_frame00030.jpg
00000_frame00040.jpg
00000_frame00050.jpg
00000_frame00060.jpg
00000_frame00070.jpg
00000_frame00080.jpg
00000_frame00090.jpg


In [12]:
!ls faces_data/train/real/original | grep id4 | head -5

id4_0001_frame00000.jpg
id4_0001_frame00010.jpg
id4_0001_frame00020.jpg
id4_0001_frame00030.jpg
id4_0001_frame00040.jpg


In [5]:
def fft_preprocess(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    f = np.fft.fft2(gray)
    f_shifted = np.fft.fftshift(f)
    magnitude = np.abs(f_shifted)
    log_magnitude = np.log1p(magnitude)
    norm = (log_magnitude - log_magnitude.min()) / (log_magnitude.max() - log_magnitude.min())
    norm_3ch = np.stack([norm, norm, norm], axis=-1)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    fft_input = (norm_3ch - mean) / std
    return fft_input

In [6]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset

class CelebDFDataset(Dataset):
    def __init__(self, manifest, split, compression_levels, root_dir='faces_data'):
        self.root_dir = root_dir
        self.compression_levels = compression_levels

        # Filter manifest to just this split (train/val/test)
        split_df = manifest[manifest['split'] == split]

        # Build a list of (image_path, label) for every matching frame
        self.samples = []

        for _, row in split_df.iterrows():
            video_name = os.path.splitext(os.path.basename(row['video_path']))[0]
            label = row['label']
            class_folder = 'real' if label == 1 else 'fake'

            for comp in compression_levels:
                folder = os.path.join(root_dir, split, class_folder, comp)
                if not os.path.isdir(folder):
                    continue
                for fname in os.listdir(folder):
                    if fname.startswith(video_name + '_frame'):
                        self.samples.append((os.path.join(folder, fname), label))

        print(f"[{split}] compression={compression_levels}: {len(self.samples)} image samples found")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))  # safety, should already be 224x224

        # RGB tensor (ImageNet normalized)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        rgb_norm = (img / 255.0 - mean) / std
        rgb_tensor = torch.tensor(rgb_norm, dtype=torch.float32).permute(2, 0, 1)

        # FFT tensor
        fft_img = fft_preprocess(img)
        fft_tensor = torch.tensor(fft_img, dtype=torch.float32).permute(2, 0, 1)

        label_tensor = torch.tensor(label, dtype=torch.float32)

        return rgb_tensor, fft_tensor, label_tensor

In [7]:
train_dataset = CelebDFDataset(manifest, split='train', compression_levels=['original'])

[train] compression=['original']: 0 image samples found


In [16]:
!wget https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv

--2026-07-01 12:02:17--  https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63103 (62K) [text/plain]
Saving to: ‘split_manifest.csv.2’

split_manifest.csv. 100%[===================>]  61.62K  --.-KB/s    in 0.007s  

2026-07-01 12:02:17 (8.05 MB/s) - ‘split_manifest.csv.2’ saved [63103/63103]



In [17]:
import pandas as pd
manifest = pd.read_csv('split_manifest.csv')
print(manifest.shape)
manifest.head()

(1203, 4)


,video_path,label,source,split
0,YouTube-real/00000.mp4,1,youtube-real,train
1,Celeb-real/id4_0001.mp4,1,celeb-real,train
2,Celeb-real/id10_0007.mp4,1,celeb-real,train
3,YouTube-real/00079.mp4,1,youtube-real,train
4,Celeb-real/id11_0004.mp4,1,celeb-real,train


In [18]:
val_dataset = CelebDFDataset(manifest, split='val', compression_levels=['original'])

[val] compression=['original']: 7198 image samples found


In [19]:
rgb_sample, fft_sample, label_sample = train_dataset[0]
print(rgb_sample.shape, fft_sample.shape, label_sample)

torch.Size([3, 224, 224]) torch.Size([3, 224, 224]) tensor(1.)


In [20]:
!pip install -q timm

In [8]:
import torch.nn as nn
import timm

class DualStreamModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.fft_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(2560, 1)

    def forward(self, rgb_input, fft_input):
        rgb_feat = self.rgb_branch(rgb_input)
        fft_feat = self.fft_branch(fft_input)
        fused = torch.cat([rgb_feat, fft_feat], dim=1)
        dropped = self.dropout(fused)
        logit = self.classifier(dropped)
        return logit

In [22]:
print(manifest.shape)

(1203, 4)


In [23]:
def fft_preprocess(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    f = np.fft.fft2(gray)
    f_shifted = np.fft.fftshift(f)
    magnitude = np.abs(f_shifted)
    log_magnitude = np.log1p(magnitude)
    norm = (log_magnitude - log_magnitude.min()) / (log_magnitude.max() - log_magnitude.min())
    norm_3ch = np.stack([norm, norm, norm], axis=-1)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    fft_input = (norm_3ch - mean) / std
    return fft_input

In [24]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset

class CelebDFDataset(Dataset):
    def __init__(self, manifest, split, compression_levels, root_dir='faces_data'):
        self.root_dir = root_dir
        self.compression_levels = compression_levels
        split_df = manifest[manifest['split'] == split]
        self.samples = []
        for _, row in split_df.iterrows():
            video_name = os.path.splitext(os.path.basename(row['video_path']))[0]
            label = row['label']
            class_folder = 'real' if label == 1 else 'fake'
            for comp in compression_levels:
                folder = os.path.join(root_dir, split, class_folder, comp)
                if not os.path.isdir(folder):
                    continue
                for fname in os.listdir(folder):
                    if fname.startswith(video_name + '_frame'):
                        self.samples.append((os.path.join(folder, fname), label))
        print(f"[{split}] compression={compression_levels}: {len(self.samples)} image samples found")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        rgb_norm = (img / 255.0 - mean) / std
        rgb_tensor = torch.tensor(rgb_norm, dtype=torch.float32).permute(2, 0, 1)
        fft_img = fft_preprocess(img)
        fft_tensor = torch.tensor(fft_img, dtype=torch.float32).permute(2, 0, 1)
        label_tensor = torch.tensor(label, dtype=torch.float32)
        return rgb_tensor, fft_tensor, label_tensor

In [25]:
train_dataset = CelebDFDataset(manifest, split='train', compression_levels=['original'])
val_dataset = CelebDFDataset(manifest, split='val', compression_levels=['original'])

[train] compression=['original']: 34644 image samples found
[val] compression=['original']: 7198 image samples found


In [26]:
import timm
print(timm.__version__)

1.0.27


In [9]:
import torch.nn as nn
import timm

class DualStreamModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.fft_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(2560, 1)

    def forward(self, rgb_input, fft_input):
        rgb_feat = self.rgb_branch(rgb_input)
        fft_feat = self.fft_branch(fft_input)
        fused = torch.cat([rgb_feat, fft_feat], dim=1)
        dropped = self.dropout(fused)
        logit = self.classifier(dropped)
        return logit

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [29]:
print(manifest.shape)

(1203, 4)


In [30]:
print(len(train_dataset))
print(len(val_dataset))

34644
7198


In [14]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [32]:
rgb_batch, fft_batch, label_batch = next(iter(train_loader))
print(rgb_batch.shape, fft_batch.shape, label_batch.shape)

torch.Size([32, 3, 224, 224]) torch.Size([32, 3, 224, 224]) torch.Size([32])


In [33]:
model = DualStreamModel().to(device)

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [34]:
import torch.optim as optim

criterion = torch.nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [35]:
model.train()

for batch_idx, (rgb_batch, fft_batch, label_batch) in enumerate(train_loader):
    if batch_idx >= 3:
        break

    rgb_batch = rgb_batch.to(device)
    fft_batch = fft_batch.to(device)
    label_batch = label_batch.to(device).unsqueeze(1)

    optimizer.zero_grad()
    logits = model(rgb_batch, fft_batch)
    loss = criterion(logits, label_batch)
    loss.backward()
    optimizer.step()

    print(f"Batch {batch_idx} - Loss: {loss.item():.4f}")

print("Test run complete - no errors!")

Batch 0 - Loss: 0.7184
Batch 1 - Loss: 0.7205
Batch 2 - Loss: 0.6995
Test run complete - no errors!


In [36]:
import time

num_epochs = 10
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    start_time = time.time()

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (rgb_batch, fft_batch, label_batch) in enumerate(train_loader):
        rgb_batch = rgb_batch.to(device)
        fft_batch = fft_batch.to(device)
        label_batch = label_batch.to(device).unsqueeze(1)

        optimizer.zero_grad()
        logits = model(rgb_batch, fft_batch)
        loss = criterion(logits, label_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == label_batch).sum().item()
        total += label_batch.size(0)

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(train_loader)} - Loss: {loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for rgb_batch, fft_batch, label_batch in val_loader:
            rgb_batch = rgb_batch.to(device)
            fft_batch = fft_batch.to(device)
            label_batch = label_batch.to(device).unsqueeze(1)
            logits = model(rgb_batch, fft_batch)
            loss = criterion(logits, label_batch)
            val_running_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            val_correct += (preds == label_batch).sum().item()
            val_total += label_batch.size(0)

    val_loss = val_running_loss / len(val_loader)
    val_acc = val_correct / val_total

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - start_time
    print(f"\nEpoch {epoch+1}/{num_epochs} ({elapsed/60:.1f} min) — "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    ckpt_name = f'model_baseline_epoch{epoch+1}.pth'
    torch.save(model.state_dict(), ckpt_name)
    from google.colab import files
    files.download(ckpt_name)
    print(f"Checkpoint saved: {ckpt_name}")

  Batch 0/1083 - Loss: 0.6826
  Batch 100/1083 - Loss: 0.1039
  Batch 200/1083 - Loss: 0.1021
  Batch 300/1083 - Loss: 0.0416
  Batch 400/1083 - Loss: 0.0355
  Batch 500/1083 - Loss: 0.0267
  Batch 600/1083 - Loss: 0.0355
  Batch 700/1083 - Loss: 0.0345
  Batch 800/1083 - Loss: 0.0128
  Batch 900/1083 - Loss: 0.0110
  Batch 1000/1083 - Loss: 0.0390

Epoch 1/10 (6.7 min) — Train Loss: 0.0961, Train Acc: 0.9624 | Val Loss: 0.1591, Val Acc: 0.9500


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Checkpoint saved: model_baseline_epoch1.pth
  Batch 0/1083 - Loss: 0.0028
  Batch 100/1083 - Loss: 0.0296
  Batch 200/1083 - Loss: 0.0033
  Batch 300/1083 - Loss: 0.0017
  Batch 400/1083 - Loss: 0.0070
  Batch 500/1083 - Loss: 0.0009


KeyboardInterrupt: 

In [ ]:
print(num_epochs)

In [ ]:
model = DualStreamModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
print("Model reset - ready for full training")

In [37]:
print(len(train_dataset), len(val_dataset), device)

34644 7198 cuda


In [10]:
class RGBOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(1280, 1)

    def forward(self, rgb_input, fft_input):
        rgb_feat = self.rgb_branch(rgb_input)
        dropped = self.dropout(rgb_feat)
        logit = self.classifier(dropped)
        return logit

In [39]:
model_rgb = RGBOnlyModel().to(device)
optimizer_rgb = optim.Adam(model_rgb.parameters(), lr=0.0001)
print("RGB-only model ready")

RGB-only model ready


In [40]:
import time

num_epochs = 10
history_rgb = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    start_time = time.time()

    model_rgb.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (rgb_batch, fft_batch, label_batch) in enumerate(train_loader):
        rgb_batch = rgb_batch.to(device)
        fft_batch = fft_batch.to(device)
        label_batch = label_batch.to(device).unsqueeze(1)

        optimizer_rgb.zero_grad()
        logits = model_rgb(rgb_batch, fft_batch)
        loss = criterion(logits, label_batch)
        loss.backward()
        optimizer_rgb.step()

        running_loss += loss.item()
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == label_batch).sum().item()
        total += label_batch.size(0)

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(train_loader)} - Loss: {loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    model_rgb.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for rgb_batch, fft_batch, label_batch in val_loader:
            rgb_batch = rgb_batch.to(device)
            fft_batch = fft_batch.to(device)
            label_batch = label_batch.to(device).unsqueeze(1)
            logits = model_rgb(rgb_batch, fft_batch)
            loss = criterion(logits, label_batch)
            val_running_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            val_correct += (preds == label_batch).sum().item()
            val_total += label_batch.size(0)

    val_loss = val_running_loss / len(val_loader)
    val_acc = val_correct / val_total

    history_rgb['train_loss'].append(train_loss)
    history_rgb['train_acc'].append(train_acc)
    history_rgb['val_loss'].append(val_loss)
    history_rgb['val_acc'].append(val_acc)

    elapsed = time.time() - start_time
    print(f"\nEpoch {epoch+1}/{num_epochs} ({elapsed/60:.1f} min) — "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    ckpt_name = f'model_rgb_only_epoch{epoch+1}.pth'
    torch.save(model_rgb.state_dict(), ckpt_name)
    from google.colab import files
    files.download(ckpt_name)
    print(f"Checkpoint saved: {ckpt_name}\n")

  Batch 0/1083 - Loss: 0.6801
  Batch 100/1083 - Loss: 0.2569
  Batch 200/1083 - Loss: 0.2357
  Batch 300/1083 - Loss: 0.1424
  Batch 400/1083 - Loss: 0.0088
  Batch 500/1083 - Loss: 0.0295
  Batch 600/1083 - Loss: 0.1812
  Batch 700/1083 - Loss: 0.0147
  Batch 800/1083 - Loss: 0.0178
  Batch 900/1083 - Loss: 0.0328
  Batch 1000/1083 - Loss: 0.0754

Epoch 1/10 (4.7 min) — Train Loss: 0.0960, Train Acc: 0.9618 | Val Loss: 0.1617, Val Acc: 0.9439


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Checkpoint saved: model_rgb_only_epoch1.pth

  Batch 0/1083 - Loss: 0.0066
  Batch 100/1083 - Loss: 0.0016
  Batch 200/1083 - Loss: 0.0606
  Batch 300/1083 - Loss: 0.0006
  Batch 400/1083 - Loss: 0.0142
  Batch 500/1083 - Loss: 0.0011
  Batch 600/1083 - Loss: 0.0033
  Batch 700/1083 - Loss: 0.0001
  Batch 800/1083 - Loss: 0.0240
  Batch 900/1083 - Loss: 0.0022
  Batch 1000/1083 - Loss: 0.0045

Epoch 2/10 (4.7 min) — Train Loss: 0.0131, Train Acc: 0.9954 | Val Loss: 0.1829, Val Acc: 0.9450


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Checkpoint saved: model_rgb_only_epoch9.pth

  Batch 0/1083 - Loss: 0.0008
  Batch 100/1083 - Loss: 0.0048
  Batch 200/1083 - Loss: 0.0007
  Batch 300/1083 - Loss: 0.0001
  Batch 400/1083 - Loss: 0.0000
  Batch 500/1083 - Loss: 0.0000
  Batch 600/1083 - Loss: 0.0000
  Batch 700/1083 - Loss: 0.0001
  Batch 800/1083 - Loss: 0.0000
  Batch 900/1083 - Loss: 0.0001
  Batch 1000/1083 - Loss: 0.0000

Epoch 10/10 (4.7 min) — Train Loss: 0.0036, Train Acc: 0.9989 | Val Loss: 0.3407, Val Acc: 0.9442


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Checkpoint saved: model_rgb_only_epoch10.pth



In [11]:
class FFTOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fft_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(1280, 1)

    def forward(self, rgb_input, fft_input):
        fft_feat = self.fft_branch(fft_input)
        dropped = self.dropout(fft_feat)
        logit = self.classifier(dropped)
        return logit

In [42]:
model_fft = FFTOnlyModel().to(device)
optimizer_fft = optim.Adam(model_fft.parameters(), lr=0.0001)
print("FFT-only model ready")

FFT-only model ready


In [43]:
import time

num_epochs = 10
history_fft = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    start_time = time.time()

    model_fft.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (rgb_batch, fft_batch, label_batch) in enumerate(train_loader):
        rgb_batch = rgb_batch.to(device)
        fft_batch = fft_batch.to(device)
        label_batch = label_batch.to(device).unsqueeze(1)

        optimizer_fft.zero_grad()
        logits = model_fft(rgb_batch, fft_batch)
        loss = criterion(logits, label_batch)
        loss.backward()
        optimizer_fft.step()

        running_loss += loss.item()
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == label_batch).sum().item()
        total += label_batch.size(0)

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(train_loader)} - Loss: {loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    model_fft.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for rgb_batch, fft_batch, label_batch in val_loader:
            rgb_batch = rgb_batch.to(device)
            fft_batch = fft_batch.to(device)
            label_batch = label_batch.to(device).unsqueeze(1)
            logits = model_fft(rgb_batch, fft_batch)
            loss = criterion(logits, label_batch)
            val_running_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            val_correct += (preds == label_batch).sum().item()
            val_total += label_batch.size(0)

    val_loss = val_running_loss / len(val_loader)
    val_acc = val_correct / val_total

    history_fft['train_loss'].append(train_loss)
    history_fft['train_acc'].append(train_acc)
    history_fft['val_loss'].append(val_loss)
    history_fft['val_acc'].append(val_acc)

    elapsed = time.time() - start_time
    print(f"\nEpoch {epoch+1}/{num_epochs} ({elapsed/60:.1f} min) — "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    ckpt_name = f'model_fft_only_epoch{epoch+1}.pth'
    torch.save(model_fft.state_dict(), ckpt_name)
    from google.colab import files
    files.download(ckpt_name)
    print(f"Checkpoint saved: {ckpt_name}\n")

  Batch 0/1083 - Loss: 0.6592
  Batch 100/1083 - Loss: 0.6676
  Batch 200/1083 - Loss: 0.5030
  Batch 300/1083 - Loss: 0.5359
  Batch 400/1083 - Loss: 0.3925
  Batch 500/1083 - Loss: 0.3980
  Batch 600/1083 - Loss: 0.3381
  Batch 700/1083 - Loss: 0.4551
  Batch 800/1083 - Loss: 0.3822
  Batch 900/1083 - Loss: 0.4002
  Batch 1000/1083 - Loss: 0.4711

Epoch 1/10 (4.6 min) — Train Loss: 0.4817, Train Acc: 0.7690 | Val Loss: 0.5716, Val Acc: 0.7510


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Checkpoint saved: model_fft_only_epoch1.pth

  Batch 0/1083 - Loss: 0.2196
  Batch 100/1083 - Loss: 0.4108
  Batch 200/1083 - Loss: 0.2824
  Batch 300/1083 - Loss: 0.3013
  Batch 400/1083 - Loss: 0.1675
  Batch 500/1083 - Loss: 0.2879
  Batch 600/1083 - Loss: 0.3204
  Batch 700/1083 - Loss: 0.1948
  Batch 800/1083 - Loss: 0.2960
  Batch 900/1083 - Loss: 0.2622
  Batch 1000/1083 - Loss: 0.3051

Epoch 2/10 (4.6 min) — Train Loss: 0.3206, Train Acc: 0.8631 | Val Loss: 0.5593, Val Acc: 0.7562


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Checkpoint saved: model_fft_only_epoch9.pth

  Batch 0/1083 - Loss: 0.0089
  Batch 100/1083 - Loss: 0.0086
  Batch 200/1083 - Loss: 0.0261
  Batch 300/1083 - Loss: 0.0104
  Batch 400/1083 - Loss: 0.0149
  Batch 500/1083 - Loss: 0.0106
  Batch 600/1083 - Loss: 0.0462
  Batch 700/1083 - Loss: 0.0027
  Batch 800/1083 - Loss: 0.0044
  Batch 900/1083 - Loss: 0.0052
  Batch 1000/1083 - Loss: 0.0243

Epoch 10/10 (4.7 min) — Train Loss: 0.0302, Train Acc: 0.9887 | Val Loss: 1.4237, Val Acc: 0.7567


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Checkpoint saved: model_fft_only_epoch10.pth



In [44]:
def get_frame_probability(image, model_path='model_baseline_epoch7.pth'):
    """
    Takes a face image (numpy array, RGB, any size) and returns
    a single float between 0 and 1 representing probability of being REAL.
    (1 = real, 0 = fake, consistent with dataset labeling convention)

    Args:
        image: numpy array, RGB format, any size (will be resized to 224x224)
        model_path: path to the .pth weights file to use

    Returns:
        float: probability between 0 and 1
    """
    # Load model
    loaded_model = DualStreamModel().to(device)
    loaded_model.load_state_dict(torch.load(model_path, map_location=device))
    loaded_model.eval()

    # Prepare image
    img = cv2.resize(image, (224, 224))
    if len(img.shape) == 2:  # if grayscale, convert to RGB
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    # RGB tensor
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    rgb_norm = (img / 255.0 - mean) / std
    rgb_tensor = torch.tensor(rgb_norm, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

    # FFT tensor
    fft_img = fft_preprocess(img)
    fft_tensor = torch.tensor(fft_img, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

    # Get prediction
    with torch.no_grad():
        logit = loaded_model(rgb_tensor, fft_tensor)
        probability = torch.sigmoid(logit).item()

    return probability

In [47]:
# Test on one real image
import cv2
test_img = cv2.imread('faces_data/train/real/original/00000_frame00000.jpg')
test_img = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

prob = get_frame_probability(test_img)
print(f"Probability of being REAL: {prob:.4f}")
print(f"Decision: {'REAL' if prob > 0.5 else 'FAKE'}")

Probability of being REAL: 1.0000
Decision: REAL


In [46]:
from google.colab import files
uploaded = files.upload()

Saving model_baseline_epoch7.pth to model_baseline_epoch7.pth


In [15]:
!ls faces_data/train/real/original | head -5

ls: cannot access 'faces_data/train/real/original': No such file or directory


In [16]:
!unzip -o celeb-df-v2-faces.zip -d faces_data
!ls faces_data

unzip:  cannot find or open celeb-df-v2-faces.zip, celeb-df-v2-faces.zip.zip or celeb-df-v2-faces.zip.ZIP.
ls: cannot access 'faces_data': No such file or directory


In [17]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!echo KGAT_ea2cc5f42eab7752c7e422757de57a92 > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token
!kaggle datasets download -d niranjana1310/celeb-df-v2-faces
!unzip -o celeb-df-v2-faces.zip -d faces_data
!ls faces_data

Streaming output truncated to the last 5000 lines.
  inflating: faces_data/val/real/crf35/00023_frame00150.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00160.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00170.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00180.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00190.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00200.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00210.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00220.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00230.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00240.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00250.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00260.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00270.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00280.jpg  
  inflating: faces_data/val/real/crf35/00023_frame00290.jpg  
  inflating: faces_

In [18]:
!wget https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv

--2026-07-03 03:17:06--  https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63103 (62K) [text/plain]
Saving to: ‘split_manifest.csv.1’

split_manifest.csv. 100%[===================>]  61.62K  --.-KB/s    in 0.001s  

2026-07-03 03:17:07 (79.9 MB/s) - ‘split_manifest.csv.1’ saved [63103/63103]



In [19]:
import pandas as pd
manifest = pd.read_csv('split_manifest.csv')
print(manifest.shape)

(1203, 4)


In [20]:
import cv2
import numpy as np

def fft_preprocess(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    f = np.fft.fft2(gray)
    f_shifted = np.fft.fftshift(f)
    magnitude = np.abs(f_shifted)
    log_magnitude = np.log1p(magnitude)
    norm = (log_magnitude - log_magnitude.min()) / (log_magnitude.max() - log_magnitude.min())
    norm_3ch = np.stack([norm, norm, norm], axis=-1)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    fft_input = (norm_3ch - mean) / std
    return fft_input

In [21]:
import os
import torch
from torch.utils.data import Dataset

class CelebDFDataset(Dataset):
    def __init__(self, manifest, split, compression_levels, root_dir='faces_data'):
        self.root_dir = root_dir
        self.compression_levels = compression_levels
        split_df = manifest[manifest['split'] == split]
        self.samples = []
        for _, row in split_df.iterrows():
            video_name = os.path.splitext(os.path.basename(row['video_path']))[0]
            label = row['label']
            class_folder = 'real' if label == 1 else 'fake'
            for comp in compression_levels:
                folder = os.path.join(root_dir, split, class_folder, comp)
                if not os.path.isdir(folder):
                    continue
                for fname in os.listdir(folder):
                    if fname.startswith(video_name + '_frame'):
                        self.samples.append((os.path.join(folder, fname), label))
        print(f"[{split}] compression={compression_levels}: {len(self.samples)} image samples found")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        rgb_norm = (img / 255.0 - mean) / std
        rgb_tensor = torch.tensor(rgb_norm, dtype=torch.float32).permute(2, 0, 1)
        fft_img = fft_preprocess(img)
        fft_tensor = torch.tensor(fft_img, dtype=torch.float32).permute(2, 0, 1)
        label_tensor = torch.tensor(label, dtype=torch.float32)
        return rgb_tensor, fft_tensor, label_tensor

In [22]:
train_dataset = CelebDFDataset(manifest, split='train', compression_levels=['original'])
val_dataset = CelebDFDataset(manifest, split='val', compression_levels=['original'])

[train] compression=['original']: 34644 image samples found
[val] compression=['original']: 7198 image samples found


In [23]:
import torch.nn as nn
import timm

class DualStreamModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.fft_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(2560, 1)

    def forward(self, rgb_input, fft_input):
        rgb_feat = self.rgb_branch(rgb_input)
        fft_feat = self.fft_branch(fft_input)
        fused = torch.cat([rgb_feat, fft_feat], dim=1)
        dropped = self.dropout(fused)
        logit = self.classifier(dropped)
        return logit

class RGBOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(1280, 1)

    def forward(self, rgb_input, fft_input):
        rgb_feat = self.rgb_branch(rgb_input)
        dropped = self.dropout(rgb_feat)
        logit = self.classifier(dropped)
        return logit

class FFTOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fft_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(1280, 1)

    def forward(self, rgb_input, fft_input):
        fft_feat = self.fft_branch(fft_input)
        dropped = self.dropout(fft_feat)
        logit = self.classifier(dropped)
        return logit

In [24]:
import torch.optim as optim
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

cuda


In [25]:
model = DualStreamModel().to(device)
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
print("Ready")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Ready


In [26]:
def get_frame_probability(image, model_path='model_baseline_epoch7.pth'):
    loaded_model = DualStreamModel().to(device)
    loaded_model.load_state_dict(torch.load(model_path, map_location=device))
    loaded_model.eval()

    img = cv2.resize(image, (224, 224))
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    rgb_norm = (img / 255.0 - mean) / std
    rgb_tensor = torch.tensor(rgb_norm, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

    fft_img = fft_preprocess(img)
    fft_tensor = torch.tensor(fft_img, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

    with torch.no_grad():
        logit = loaded_model(rgb_tensor, fft_tensor)
        probability = torch.sigmoid(logit).item()

    return probability

In [27]:
from google.colab import files
uploaded = files.upload()

Saving model_baseline_epoch7.pth to model_baseline_epoch7.pth


In [28]:
test_img = cv2.imread('faces_data/train/real/original/00000_frame00000.jpg')
test_img = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

prob = get_frame_probability(test_img)
print(f"Probability of being REAL: {prob:.4f}")
print(f"Decision: {'REAL' if prob > 0.5 else 'FAKE'}")

Probability of being REAL: 1.0000
Decision: REAL


In [29]:
fake_img = cv2.imread('faces_data/train/fake/original/id0_id16_0006_frame00000.jpg')
fake_img = cv2.cvtColor(fake_img, cv2.COLOR_BGR2RGB)

prob = get_frame_probability(fake_img)
print(f"Probability of being REAL: {prob:.4f}")
print(f"Decision: {'REAL' if prob > 0.5 else 'FAKE'}")

error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


In [30]:
!ls faces_data/train/fake/original | head -10

id0_id1_0000_frame00000.jpg
id0_id1_0000_frame00010.jpg
id0_id1_0000_frame00020.jpg
id0_id1_0000_frame00030.jpg
id0_id1_0000_frame00040.jpg
id0_id1_0000_frame00050.jpg
id0_id1_0000_frame00060.jpg
id0_id1_0000_frame00070.jpg
id0_id1_0000_frame00080.jpg
id0_id1_0000_frame00090.jpg


In [31]:
fake_img = cv2.imread('faces_data/train/fake/original/id0_id1_0000_frame00000.jpg')
fake_img = cv2.cvtColor(fake_img, cv2.COLOR_BGR2RGB)

prob = get_frame_probability(fake_img)
print(f"Probability of being REAL: {prob:.4f}")
print(f"Decision: {'REAL' if prob > 0.5 else 'FAKE'}")

Probability of being REAL: 0.0000
Decision: FAKE


In [32]:
!pip freeze > requirements.txt
print("requirements.txt created")
!head -30 requirements.txt

requirements.txt created
absl-py==1.4.0
accelerate==1.14.0
access==1.1.10.post3
affine==2.4.0
aiofiles==25.1.0
aiohappyeyeballs==2.6.2
aiohttp==3.14.1
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.12.0
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.14.0
anywidget==0.9.21
apsw==3.53.2.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.1
astropy-iers-data==0.2026.6.22.1.23.34
astunparse==1.6.3
atpublic==5.1


In [33]:
requirements = """torch==2.6.0
torchvision==0.21.0
timm==1.0.27
opencv-python-headless==4.13.0.33
numpy==1.26.4
pandas==2.2.2
Pillow==11.2.1
scikit-learn==1.6.1
matplotlib==3.10.0
kaggle==1.7.4
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("Clean requirements.txt created")
print(requirements)

Clean requirements.txt created
torch==2.6.0
torchvision==0.21.0
timm==1.0.27
opencv-python-headless==4.13.0.33
numpy==1.26.4
pandas==2.2.2
Pillow==11.2.1
scikit-learn==1.6.1
matplotlib==3.10.0
kaggle==1.7.4



In [34]:
from google.colab import files
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>